In [1]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CreditCardFraudDetection-Streaming") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark Session Created Successfully!")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/28 21:39:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session Created Successfully!


26/03/28 21:39:27 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
from pyspark.sql.types import *

schema = StructType([
    StructField("_c0", IntegerType(), True),
    StructField("trans_date_trans_time", TimestampType(), True),
    StructField("cc_num", LongType(), True),
    StructField("merchant", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amt", DoubleType(), True),
    StructField("first", StringType(), True),
    StructField("last", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("street", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("zip", IntegerType(), True),
    StructField("lat", DoubleType(), True),
    StructField("long", DoubleType(), True),
    StructField("city_pop", IntegerType(), True),
    StructField("job", StringType(), True),
    StructField("dob", DateType(), True),
    StructField("trans_num", StringType(), True),
    StructField("unix_time", IntegerType(), True),
    StructField("merch_lat", DoubleType(), True),
    StructField("merch_long", DoubleType(), True),
    StructField("is_fraud", IntegerType(), True),
    StructField("split", StringType(), True)
])

print("Schema defined!")

Schema defined!


In [3]:
# Read streaming data as a stream
df_stream = spark.readStream \
    .option("header", True) \
    .schema(schema) \
    .csv("../data/streaming_data/")

print("Streaming source ready!")
print("Is Streaming:", df_stream.isStreaming)

Streaming source ready!
Is Streaming: True


In [4]:
# Write stream to console - shows transactions as they come in
query1 = df_stream.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("numRows", 20) \
    .option("truncate", False) \
    .start()

query1.awaitTermination(timeout=15)  # runs for 15 seconds
print("Query 1 complete!")

-------------------------------------------
Batch: 0
-------------------------------------------
+------+---------------------+-------------------+---------------------------------------+-------------+------+---------+--------+------+--------------------------------+-----------------+-----+-----+-----------------+------------------+--------+----------------------------------+----------+--------------------------------+----------+------------------+-------------------+--------+------+
|_c0   |trans_date_trans_time|cc_num             |merchant                               |category     |amt   |first    |last    |gender|street                          |city             |state|zip  |lat              |long              |city_pop|job                               |dob       |trans_num                       |unix_time |merch_lat         |merch_long         |is_fraud|split |
+------+---------------------+-------------------+---------------------------------------+-------------+------+--------

In [5]:
from pyspark.sql.functions import col

# Count fraud vs non-fraud as stream flows in
df_fraud_count = df_stream \
    .groupBy("is_fraud") \
    .count()

query2 = df_fraud_count.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

query2.awaitTermination(timeout=15)
print("Query 2 complete!")

-------------------------------------------
Batch: 0
-------------------------------------------
+--------+------+
|is_fraud| count|
+--------+------+
|       1|   924|
|       0|276936|
+--------+------+

Query 2 complete!


In [6]:
# Fraud transactions by category as stream flows in
df_fraud_category = df_stream \
    .filter(col("is_fraud") == 1) \
    .groupBy("category") \
    .count() \
    .orderBy("count", ascending=False)

query3 = df_fraud_category.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

query3.awaitTermination(timeout=15)
print("Query 3 complete!")

-------------------------------------------
Batch: 0
-------------------------------------------
+--------------+-----+
|      category|count|
+--------------+-----+
|  shopping_net|  224|
|   grocery_pos|  215|
|      misc_net|  115|
|  shopping_pos|   89|
| gas_transport|   81|
|      misc_pos|   30|
| personal_care|   29|
|     kids_pets|   27|
|          home|   27|
|   food_dining|   20|
|   grocery_net|   19|
|health_fitness|   19|
| entertainment|   15|
|        travel|   14|
+--------------+-----+

Query 3 complete!


In [ ]:
# Flag high value fraud transactions (amt > 500)
df_high_fraud = df_stream \
    .filter((col("is_fraud") == 1) & (col("amt") > 500)) \
    .select("trans_num", "amt", "category", "merchant", "is_fraud")

query4 = df_high_fraud.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("numRows", 20) \
    .start()

query4.awaitTermination(timeout=15)
print("Query 4 complete!")

-------------------------------------------
Batch: 0
-------------------------------------------
+--------------------+-------+------------+--------------------+--------+
|           trans_num|    amt|    category|            merchant|is_fraud|
+--------------------+-------+------------+--------------------+--------+
|5c70149dfd54b264b...| 847.92|shopping_pos|     fraud_Lynch Ltd|       1|
|f4f1d5407a78d18d7...|1148.36|shopping_net|fraud_Langworth, ...|       1|
|4c4958e5c21efd9e4...|1104.51|shopping_net|fraud_Reichert, H...|       1|
|277aee1058f9ea771...| 750.14|    misc_net|fraud_Moore, Dibb...|       1|
|d553cb71531da1951...| 959.44|shopping_pos|fraud_Macejkovic-...|       1|
|71acb10d85e7e2902...| 710.78|shopping_pos|fraud_Macejkovic-...|       1|
|25f9dce4d55b3d6b2...| 715.39|    misc_net|fraud_Kerluke, Ke...|       1|
|749167bd16bb88a37...|1115.75|shopping_net|fraud_Streich, Di...|       1|
|0897f5a7bc7bc283f...| 776.66|    misc_net|  fraud_Kris-Weimann|       1|
|3e1750a681719b

26/03/28 22:40:13 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterE

In [ ]:
# Stop all active streaming queries cleanly
for q in spark.streams.active:
    q.stop()

spark.stop()
print("All queries stopped. Spark session closed cleanly.")